In [ ]:
from pathlib import Path
import numpy as np
from collections import defaultdict, Counter
import matplotlib.pyplot as plt
import pandas as pd
import tqdm, os, itertools, allel, subprocess

In [ ]:
os.chdir('/home/tymekpieszko/projects/RotSpec/classical/')
import importlib
mode = "silwood" # silwood or total
import unphased_local as uph

importlib.reload(uph)

In [ ]:
def filt_samples(genos, samples, targets):
    mask = [i for i, s in enumerate(samples) if s in targets]
    return genos[:,mask]
def filt_biallelic(genos, pos, scaffolds):
    ac = genos.count_alleles()
    mask = ac.is_biallelic_01()
    return genos[mask], pos[mask], scaffolds[mask]

In [ ]:
# Note that SNP-only VCFs are used here.

vcf_rm = '../data/vcfs_total/Rmag_polar.SNPs.vcf.gz'
vcf_rs = '../data/vcfs_total/Rsoc_polar.SNPs.vcf.gz'
callset_rm = allel.read_vcf(vcf_rm, fields=['samples', 'variants/CHROM', 'variants/POS', 'calldata/GT'])
callset_rs = allel.read_vcf(vcf_rs, fields=['samples', 'variants/CHROM', 'variants/POS', 'calldata/GT'])
############
### Rmag ###
############
genos_rm = allel.GenotypeArray(callset_rm['calldata/GT'])
pos_rm =  callset_rm['variants/POS']
scaffolds_rm = callset_rm['variants/CHROM']
genos_rm_SIL = filt_samples(genos_rm, callset_rm['samples'], uph.rm_SIL_samples)
genos_rm_SIL, pos_rm_SIL, scaffolds_rm_SIL = filt_biallelic(genos_rm_SIL, pos_rm, scaffolds_rm)
genos_rm_other = filt_samples(genos_rm, callset_rm['samples'], uph.rm_other_samples)
genos_rm_other, pos_rm_other, scaffolds_rm_other = filt_biallelic(genos_rm_other, pos_rm, scaffolds_rm)
############
### Rsoc ###
############
genos_rs = allel.GenotypeArray(callset_rs['calldata/GT'])
pos_rs =  callset_rs['variants/POS']
scaffolds_rs = callset_rs['variants/CHROM']
genos_rs_SIL = filt_samples(genos_rs, callset_rs['samples'], uph.rs_SIL_samples)
genos_rs_SIL, pos_rs_SIL, scaffolds_rs_SIL = filt_biallelic(genos_rs_SIL, pos_rs, scaffolds_rs)
genos_rs_other = filt_samples(genos_rs, callset_rs['samples'], uph.rs_other_samples)
genos_rs_other, pos_rs_other, scaffolds_rs_other = filt_biallelic(genos_rs_other, pos_rs, scaffolds_rs)

In [ ]:
###############################
### DIVERSITY (PURE PYTHON) ###
###############################

# pixy version below - decided against it.

In [ ]:
def calc_div_vcf(genos, pos, scaffolds, mode):
    idx = itertools.combinations(np.arange(genos.n_samples*2), r=2)
    # if mode=="all":
    #     idx = idx
    if mode=="between":
        idx = [x for x in idx if not (x[1] - x[0] == 1 and x[1] % 2 == 1)]
    if mode=="within":
        idx = [(i, i+1) for i in range(0, genos.n_samples*2, 2)]
    total = {}
    for i1,i2 in tqdm.tqdm(idx):
        h1 = genos[:,i1//2,i1%2]
        h2 = genos[:,i2//2,i2%2]
        genos_i = allel.GenotypeArray([[x] for x in np.stack((h1,h2), axis=1)])
        hets_i = []
        for genos_ik, pos_k in uph.iter_scaffolds(genos_i, pos, scaffolds):
            hets, intervals = allel.windowed_statistic(pos_k, genos_ik, statistic=lambda x : x.is_het(allele=1).sum(), start=0, stop=pos_k[-1], size=5001)[0:2]
            hets = np.nan_to_num(hets)
            hets = hets / (intervals[:,1]-intervals[:,0])
            hets_i.extend(hets[:-1])
        if mode=="within":
            total[i1//2] = hets_i
        else:
            total[f"{i1}_{i2}"] = hets_i 
    return total

In [ ]:
within_rm_SIL = calc_div_vcf(genos_rm_SIL, pos_rm_SIL, scaffolds_rm_SIL, mode="within")
within_rs_SIL = calc_div_vcf(genos_rs_SIL, pos_rs_SIL, scaffolds_rs_SIL, mode="within")
between_rm_SIL = calc_div_vcf(genos_rm_SIL, pos_rm_SIL, scaffolds_rm_SIL, mode="between")
between_rs_SIL = calc_div_vcf(genos_rs_SIL, pos_rs_SIL, scaffolds_rs_SIL, mode="between")
# all_rm_SIL = calc_div_vcf(genos_rm_SIL, pos_rm_SIL, scaffolds_rm_SIL, mode="all")
# all_rs_SIL = calc_div_vcf(genos_rs_SIL, pos_rs_SIL, scaffolds_rs_SIL, mode="all")

within_rm_other = calc_div_vcf(genos_rm_other, pos_rm_other, scaffolds_rm_other, mode="within")
within_rs_other = calc_div_vcf(genos_rs_other, pos_rs_other, scaffolds_rs_other, mode="within")
between_rm_other = calc_div_vcf(genos_rm_other, pos_rm_other, scaffolds_rm_other, mode="between")
between_rs_other = calc_div_vcf(genos_rs_other, pos_rs_other, scaffolds_rs_other, mode="between")
# all_rm_other = calc_div_vcf(genos_rm_other, pos_rm_other, scaffolds_rm_other, mode="all")
# all_rs_other = calc_div_vcf(genos_rs_other, pos_rs_other, scaffolds_rs_other, mode="all")

In [ ]:
within_rm_all_mean     = np.mean([np.mean(vals) for vals in (within_rm_SIL | within_rm_other).values()])
within_rs_all_mean     = np.mean([np.mean(vals) for vals in (within_rs_SIL | within_rs_other).values()])
within_rm_SIL_mean     = np.mean([np.mean(vals) for vals in within_rm_SIL.values()])
within_rs_SIL_mean     = np.mean([np.mean(vals) for vals in within_rs_SIL.values()])
between_rm_SIL_mean    = np.mean([np.mean(vals) for vals in between_rm_SIL.values()])
between_rs_SIL_mean    = np.mean([np.mean(vals) for vals in between_rs_SIL.values()])
within_rm_other_mean  = np.mean([np.mean(vals) for vals in within_rm_other.values()])
within_rs_other_mean  = np.mean([np.mean(vals) for vals in within_rs_other.values()])
between_rm_other_mean = np.mean([np.mean(vals) for vals in between_rm_other.values()])
between_rs_other_mean = np.mean([np.mean(vals) for vals in between_rs_other.values()])



print("##########")
print(within_rm_all_mean*100)
print(within_rs_all_mean*100)
print("##########")
print(within_rm_SIL_mean*100)
print(within_rs_SIL_mean*100)
print("##########")
print(between_rm_SIL_mean*100)
print(between_rs_SIL_mean*100)
print("##########")
print(within_rm_other_mean*100)
print(within_rs_other_mean*100)
print("##########")
print(between_rm_other_mean*100)
print(between_rs_other_mean*100)
print("##########")


In [ ]:
################
### NEW PLOT ###
################
lw = 4
order = lambda callset, targets: [s for s in callset['samples'] if s in targets]
blocks = [(order(callset_rm, uph.rm_SIL_samples), within_rm_SIL, between_rm_SIL_mean, uph.color_rm),
          (order(callset_rm, uph.rm_other_samples), within_rm_other, between_rm_other_mean, uph.color_rm),
          (order(callset_rs, uph.rs_SIL_samples), within_rs_SIL, between_rs_SIL_mean, uph.color_rs),
          (order(callset_rs, uph.rs_other_samples), within_rs_other, between_rs_other_mean, uph.color_rs)]
if mode == "silwood":
    blocks = [blocks[0], blocks[2]]
plt.figure(figsize=(6, 10 if mode == "total" else 8))
ticks, labels, y = [], [], 0
for k, (samples, within, between, color) in enumerate(blocks):
    if k > 0 and color != blocks[k - 1][3]:
        y += 1
    elif k > 0:
        plt.axhline(y - 0.5, linestyle="-", color="gray", linewidth=1.0)
    ys = y + np.arange(len(samples))
    for yi, h in zip(ys, within.values()):
        m = np.nanmean(h)
        plt.errorbar(
            m, yi,
            xerr=[[m - np.nanquantile(h, 0.25)], [np.nanquantile(h, 0.75) - m]],
            marker='o', linestyle='',
            color=color,
            ecolor=color,
            markersize=10,
            elinewidth=lw,
            capsize=8,
        )
    plt.vlines(x=between, ymin=ys[0] - 0.5, ymax=ys[-1] + 0.5, color=color, linestyle=':', linewidth=lw)
    ticks.extend(ys)
    labels.extend(samples)
    y = ys[-1] + 1
plt.xlim(-0.0001,0.006)
plt.yticks(ticks=ticks, labels=labels, fontsize=18)
plt.xticks(ticks=np.arange(0,0.0062,0.002), labels=[0.0, 0.2, 0.4, 0.6], fontsize=18)
plt.tight_layout()
Path("./plot_scripts/plots/het").mkdir(parents=True, exist_ok=True)
plt.savefig(f"./plot_scripts/plots/het/het.{mode}.png")
plt.show()

In [ ]:
########################
### DIVERSITY (PIXY) ###
########################

# Decided against it, was VERY slow!
# Needs pop files with two columns:
# sample name repeated twice
# (each sample is in its own population).

In [ ]:
# Note that this uses variant + invariant VCFs

# sp_lst = ["Rmag", "Rsoc"]
# for sp in sp_lst:
#     pops = f"{sp}_pops.txt"
#     subprocess.run(["pixy", "--stats", "pi", "dxy", "--vcf", f"../data/vcfs_total/{sp}_phased.vcf.gz",
#                     "--populations", str(pops), "--window_size", "5000", "--n_cores", "18",
#                     "--output_folder", ".", "--output_prefix", sp], check=True)

In [ ]:
###########
### FIS ###
###########

In [ ]:
fis_rm = allel.inbreeding_coefficient(genos_rm)
fis_rs = allel.inbreeding_coefficient(genos_rs)
fis_rm_SIL = allel.inbreeding_coefficient(genos_rm_SIL)
fis_rs_SIL = allel.inbreeding_coefficient(genos_rs_SIL)
fis_rm = fis_rm[~np.isnan(fis_rm)]
fis_rs = fis_rs[~np.isnan(fis_rs)]
fis_rm_SIL = fis_rm_SIL[~np.isnan(fis_rm_SIL)]
fis_rs_SIL = fis_rs_SIL[~np.isnan(fis_rs_SIL)]

In [ ]:
################
### NEW PLOT ###
################

if mode == "silwood":
    fis, positions, labels, colors = [fis_rm_SIL, fis_rs_SIL], [1.5, 2.5], ["Rmag", "Rsoc"], [uph.color_rm, uph.color_rs]
else:
    fis, positions, labels, colors = [fis_rm, fis_rm_SIL, fis_rs, fis_rs_SIL], [0.7, 1.3, 2.7, 3.3], ["Total", "SIL"]*2, [uph.color_rm]*2 + [uph.color_rs]*2
plt.figure(figsize=(6,5))
ax = plt.boxplot(x=fis, orientation="vertical", positions=positions, widths=0.5, patch_artist=True)
print("Medians:")
print([np.median(f) for f in fis])
print("Prop > 0:")
print([sum(f>0)/len(f) for f in fis])
for line in ax["medians"]:
    line.set_color("black")
for i, patch in enumerate(ax['boxes']):
    patch.set_facecolor(colors[i])
plt.axhline(0, linewidth=1, color="black", linestyle="--")
plt.xticks(ticks=positions, labels=labels, fontsize=18)
plt.yticks(ticks=[-1,-0.5,0.0,0.5,1.0], fontsize=18)
plt.tight_layout()
Path("./plot_scripts/plots/fis").mkdir(parents=True, exist_ok=True)
plt.savefig(f"./plot_scripts/plots/fis/fis.{mode}.png")
plt.show()

In [ ]:
##############################
### LINKAGE DISEQUILIBRIUM ###
##############################
# This is just a prototype

In [ ]:
def calc_r2(genos, pos, scaffolds, K, d_max):
    # K = number of target SNPs
    # d_max = maximum distance from target SNP
    r2_total = []
    dist_total = []
    for _ in tqdm.tqdm(range(K)):
        ##############
        target = np.random.randint(0, len(genos)-1)
        target_geno = genos[target]
        target_pos = pos[target]
        target_scaffold = scaffolds[target]
        ##############
        other = np.where(scaffolds == target_scaffold)[0]
        other_genos = genos[other]
        other_pos = pos[other]
        ### Skip if target is the only SNP at a scaffold
        if len(other) == 1:
            continue
        ##############
        target_geno_n = [target_geno.to_n_alt()]
        other_genos_n = other_genos.to_n_alt()
        r_i = np.atleast_1d(allel.rogers_huff_r_between(target_geno_n, other_genos_n))[0]
        r2_i = r_i ** 2
        dist_i = np.abs(other_pos - target_pos)
        mask = (dist_i!=0) & (dist_i<d_max)
        r2_total.extend(r2_i[mask])
        dist_total.extend(dist_i[mask])
    # Sort by distance
    r2_total = np.array(r2_total)
    dist_total = np.array(dist_total)
    sort_idx = np.argsort(dist_total)
    return r2_total[sort_idx], dist_total[sort_idx]

r2, dist = calc_r2(genos_rm, pos_rm, scaffolds_rm, K=10000, d_max=6000)
# print(len(r2))
# print(dist)